# 8. Building Interactive Applications with Databricks Apps

Databricks Apps lets you build and deploy production-quality web applications directly from your data platform. Turn your data insights, ML models, and AI agents into interactive applications that your team or customers can use—without managing separate infrastructure.

## What are Databricks Apps?

**Databricks Apps** enables you to deploy interactive web applications using popular Python frameworks like Streamlit, Gradio, Dash, and Flask. These apps run natively on Databricks, with built-in access to your data, models, and security.

### Common Use Cases:
* Internal tools and dashboards for business teams
* ML model inference interfaces
* AI agent chatbots and assistants
* Data exploration and analysis tools
* Customer-facing data applications

### Key Benefits:
* No separate infrastructure to manage
* Automatic Unity Catalog integration
* Built-in authentication and governance
* Scale from prototype to production seamlessly

## Supported Frameworks:
* Python frameworks: Streamlit, Dash, and Gradio
* Node.js frameworks: React, Angular, Svelte, and Express

## Databricks Apps Overview (Video)
[![Video Thumbnail](https://img.youtube.com/vi/Equ7PBeM-Mw/0.jpg)](https://www.youtube.com/watch?v=Equ7PBeM-Mw)

📖 **Resource:** [Databricks Apps Documentation](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)

## Method 1: Creating Apps from the UI

The quickest way to deploy an app is through the Databricks workspace UI.

### To Create Your First App:
1. In the left navigation bar, click **+ New** > **App**
2. Choose your framework (Streamlit, Gradio, Dash, or Flask)
3. Select a compute resource or create a new one
4. Write your app code in the built-in editor or link to a repository
5. Click **Deploy** to publish your app

Your app will be accessible via a secure URL that you can share with your team.

📖 **Resource:** [Create and deploy apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/get-started)

## Method 2: Building Apps with Code (Streamlit)

For developers who prefer working in notebooks or IDEs, you can build and deploy apps programmatically. **Streamlit** is a popular framework for data apps due to its simplicity. 

Below is a minimal example that queries a Unity Catalog table and displays interactive visualizations.

Below is a sample app.py file. Note that the `DATABRICKS_WAREHOUSE_ID` must be set in a separate `app.yaml` file. A sample app.yaml file can look like the following:

<br>

```
command: [
  "streamlit", 
  "run",
  "app.py"
]

env:
  - name: "DATABRICKS_WAREHOUSE_ID"
    valueFrom: "sql-warehouse"
  - name: STREAMLIT_BROWSER_GATHER_USAGE_STATS
    value: "false"
```

In [0]:
# Example: Simple Streamlit app that queries data and displays it
# Save this as app.py in your workspace or repository

import os
from databricks import sql
from databricks.sdk.core import Config
import streamlit as st
import pandas as pd

# Ensure environment variable is set correctly
assert os.getenv('DATABRICKS_WAREHOUSE_ID'), "DATABRICKS_WAREHOUSE_ID must be set in app.yaml."

def sqlQuery(query: str) -> pd.DataFrame:
    cfg = Config() # Pull environment variables for auth
    with sql.connect(
        server_hostname=cfg.host,
        http_path=f"/sql/1.0/warehouses/{os.getenv('DATABRICKS_WAREHOUSE_ID')}",
        credentials_provider=lambda: cfg.authenticate
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            return cursor.fetchall_arrow().to_pandas()

st.set_page_config(layout="wide")

@st.cache_data(ttl=30)  # only re-query if it's been 30 seconds
def getData():
    # This example query depends on the nyctaxi data set in Unity Catalog, see https://docs.databricks.com/en/discover/databricks-datasets.html for details
    return sqlQuery("select * from samples.nyctaxi.trips limit 5000")

data = getData()

st.header("Taxi fare distribution !!! :)")
col1, col2 = st.columns([3, 1])
with col1:
    st.scatter_chart(data=data, height=400, width=700, y="fare_amount", x="trip_distance")
with col2:
    st.subheader("Predict fare")
    pickup = st.text_input("From (zipcode)", value="10003")
    dropoff = st.text_input("To (zipcode)", value="11238")
    d = data[(data['pickup_zip'] == int(pickup)) & (data['dropoff_zip'] == int(dropoff))]
    st.write(f"# **${d['fare_amount'].mean() if len(d) > 0 else 99:.2f}**")

st.dataframe(data=data, height=600, use_container_width=True)

You can also add all of your requirements in a requirements.txt file. Sample of requirements:

<br>

```
streamlit==1.38.0
```

### Deploying Your App

Once you've written your app code:

1. Save it to a file (e.g., `app.py`) in your Databricks workspace or Git repository
2. Use the UI to create a new app and point it to your file
3. Or use the Databricks CLI:

```bash
databricks apps deploy <my-app> --source-code-path <path-to-repo>
```

📖 **Resource:** [Deploy apps with the CLI](https://docs.databricks.com/en/dev-tools/databricks-apps/deploy-app-cli.html)

## Method 3: Apps with AI Agent Integration

One of the most powerful patterns is combining Databricks Apps with AI agents. This lets you build chatbots and assistants that can interact with your data, answer questions, and take actions.

Below is an example using **Gradio** to create a chatbot interface connected to a deployed AI agent.

In [0]:
# Example: Gradio chatbot app connected to an AI agent
# This app provides a chat interface for your deployed agent

import gradio as gr
import logging
import os
from model_serving_utils import query_endpoint, is_endpoint_supported

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ensure environment variable is set correctly
SERVING_ENDPOINT = os.getenv('SERVING_ENDPOINT')
assert SERVING_ENDPOINT,\
    ("Unable to determine serving endpoint to use for chatbot app. If developing locally, "
     "set the SERVING_ENDPOINT environment variable to the name of your serving endpoint. If "
     "deploying to a Databricks app, include a serving endpoint resource named "
     "'serving_endpoint' with CAN_QUERY permissions, as described in "
     "https://docs.databricks.com/aws/en/generative-ai/agent-framework/chat-app#deploy-the-databricks-app")

# Check if the endpoint is supported
endpoint_supported = is_endpoint_supported(SERVING_ENDPOINT)

def query_llm(message, history):
    """
    Query the LLM with the given message and chat history.
    `message`: str - the latest user input.
    `history`: list of dicts - OpenAI-style messages.
    """
    if not message.strip():
        return "ERROR: The question should not be empty"

    # Convert from Gradio-style history to OpenAI-style messages
    message_history = []
    for user_msg, assistant_msg in history:
        message_history.append({"role": "user", "content": user_msg})
        message_history.append({"role": "assistant", "content": assistant_msg})

    # Add the latest user message
    message_history.append({"role": "user", "content": message})

    try:
        logger.info(f"Sending request to model endpoint: {SERVING_ENDPOINT}")
        response = query_endpoint(
            endpoint_name=SERVING_ENDPOINT,
            messages=message_history,
            max_tokens=400
        )
        return response["content"]
    except Exception as e:
        logger.error(f"Error querying model: {str(e)}", exc_info=True)
        return f"Error: {str(e)}"

# Create Gradio interface based on endpoint support
if not endpoint_supported:
    # Create a simple interface showing the error message
    with gr.Blocks() as demo:
        gr.Markdown("# Databricks LLM Chatbot")
        gr.Markdown(
            f"""
            ⚠️ **Unsupported Endpoint Type**
            
            The endpoint `{SERVING_ENDPOINT}` is not compatible with this basic chatbot template.
            
            This template only supports chat completions-compatible endpoints.
            
            👉 **For a richer chatbot template** that supports all conversational endpoints on Databricks, 
            please see the [Databricks documentation](https://docs.databricks.com/aws/en/generative-ai/agent-framework/chat-app).
            """
        )
else:
    demo = gr.ChatInterface(
        fn=query_llm,
        title="Databricks LLM Chatbot",
        description=(
            "Note: this is a simple example. See "
            "[Databricks docs](https://docs.databricks.com/aws/en/generative-ai/agent-framework/chat-app) "
            "for a more comprehensive example, with support for streaming output and more."
        ),
        examples=[
            "What is machine learning?",
            "What are Large Language Models?",
            "What is Databricks?"
        ],
    )

if __name__ == "__main__":
    demo.launch()

## Advanced Patterns

### Multi-Page Applications

Streamlit supports multi-page apps out of the box. Create a `pages/` directory with multiple Python files, and Streamlit will automatically create navigation:

```
app.py                  # Main page
pages/
  1_📊_Analytics.py     # Analytics page
  2_🤖_AI_Chat.py       # AI chat page
  3_⚙️_Settings.py      # Settings page
```

### Authentication and Access Control

Apps automatically inherit Databricks authentication. You can control access using:
* Unity Catalog permissions (table/schema level)
* App-level permissions (who can view/edit the app)
* Custom logic 

📖 **Resource:** [Read more on authorization](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/auth#gsc.tab=0)

### Connecting to Unity Catalog

Apps running on Databricks have native access to Unity Catalog. Use the SQL connector to query tables securely:

```python
import os
from databricks import sql

cnx = sql.connect(
    server_hostname=os.getenv("DATABRICKS_HOST"),
    http_path=os.getenv("DATABRICKS_WAREHOUSE_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN"),
)

with cnx.cursor() as cursor:
    cursor.execute("SELECT count(*) FROM catalog.schema.table")
    print(cursor.fetchall())

```

## Example: Streamlit Database App

This example shows how to build an app that connects to a postgres database

In [0]:
# Example: Streamlit app with Postgres Database

import streamlit as st
import psycopg
import os
import time
import re
from databricks import sdk
from psycopg import sql
from psycopg_pool import ConnectionPool

# Database connection setup
workspace_client = sdk.WorkspaceClient()
postgres_password = None
last_password_refresh = 0
connection_pool = None

def refresh_oauth_token():
    """Refresh OAuth token if expired."""
    global postgres_password, last_password_refresh
    if postgres_password is None or time.time() - last_password_refresh > 900:
        print("Refreshing PostgreSQL OAuth token")
        try:
            postgres_password = workspace_client.config.oauth_token().access_token
            last_password_refresh = time.time()
        except Exception as e:
            st.error(f"❌ Failed to refresh OAuth token: {str(e)}")
            st.stop()

def get_connection_pool():
    """Get or create the connection pool."""
    global connection_pool
    if connection_pool is None:
        refresh_oauth_token()
        conn_string = (
            f"dbname={os.getenv('PGDATABASE')} "
            f"user={os.getenv('PGUSER')} "
            f"password={postgres_password} "
            f"host={os.getenv('PGHOST')} "
            f"port={os.getenv('PGPORT')} "
            f"sslmode={os.getenv('PGSSLMODE', 'require')} "
            f"application_name={os.getenv('PGAPPNAME')}"
        )
        connection_pool = ConnectionPool(conn_string, min_size=2, max_size=10)
    return connection_pool

def get_connection():
    """Get a connection from the pool."""
    global connection_pool
    
    # Recreate pool if token expired
    if postgres_password is None or time.time() - last_password_refresh > 900:
        if connection_pool:
            connection_pool.close()
            connection_pool = None
    
    return get_connection_pool().connection()

def get_schema_name():
    """Get the schema name in the format {PGAPPNAME}_schema_{PGUSER}."""
    pgappname = os.getenv("PGAPPNAME", "my_app")
    pguser = os.getenv("PGUSER", "").replace('-', '')
    return f"{pgappname}_schema_{pguser}"

def init_database():
    """Initialize database schema and table."""
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema_name = get_schema_name()
            
            cur.execute(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema_name)))
            cur.execute(sql.SQL("""
                CREATE TABLE IF NOT EXISTS {}.todos (
                    id SERIAL PRIMARY KEY,
                    task TEXT NOT NULL,
                    completed BOOLEAN DEFAULT FALSE,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """).format(sql.Identifier(schema_name)))
            conn.commit()
            return True

def add_todo(task):
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema = get_schema_name()
            cur.execute(sql.SQL("INSERT INTO {}.todos (task) VALUES (%s)").format(sql.Identifier(schema)), (task.strip(),))
            conn.commit()

def get_todos():
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema = get_schema_name()
            cur.execute(sql.SQL("SELECT id, task, completed, created_at FROM {}.todos ORDER BY created_at DESC").format(sql.Identifier(schema)))
            return cur.fetchall()

def toggle_todo(todo_id):
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema = get_schema_name()
            cur.execute(sql.SQL("UPDATE {}.todos SET completed = NOT completed WHERE id = %s").format(sql.Identifier(schema)), (todo_id,))
            conn.commit()


def delete_todo(todo_id):
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema = get_schema_name()
            cur.execute(sql.SQL("DELETE FROM {}.todos WHERE id = %s").format(sql.Identifier(schema)), (todo_id,))
            conn.commit()

@st.fragment
def display_todos():
    st.subheader("📋 Your Todos")
    
    todos = get_todos()
    
    if not todos:
        st.info("🎉 No todos yet! Add one above to get started.")
    else:
        for todo_id, task, completed, created_at in todos:
            col1, col2, col3 = st.columns([0.1, 0.7, 0.2])
            
            with col1:
                if st.checkbox("", value=completed, key=f"check_{todo_id}"):
                    if not completed:
                        toggle_todo(todo_id)
                        st.rerun(scope="fragment")
                elif completed:
                    toggle_todo(todo_id)
                    st.rerun(scope="fragment")
            
            with col2:
                st.markdown(f"~~{task}~~ ✅" if completed else task)
                st.caption(f"Created: {created_at.strftime('%Y-%m-%d %H:%M')}")
            
            with col3:
                if st.button("🗑️", key=f"delete_{todo_id}"):
                    delete_todo(todo_id)
                    st.rerun(scope="fragment")


# Streamlit UI
def main():
    st.set_page_config(
        page_title="Todo List App",
        page_icon="✅",
        layout="wide"
    )
    
    st.title("📝 Todo List App")
    st.markdown("---")
    
    # Initialize database
    if not init_database():
        st.stop()
    
    # Add new todo section
    st.subheader("➕ Add New Todo")
    with st.form("add_todo_form", clear_on_submit=True):
        new_task = st.text_input("Enter a new task:", placeholder="What do you need to do?")
        submitted = st.form_submit_button("Add Todo", type="primary")
        
        if submitted and new_task.strip():
            if add_todo(new_task.strip()):
                st.success("✅ Todo added successfully!")
    
    st.markdown("---")
    
    display_todos()

if __name__ == "__main__":
    main() 

## Using Databricks Secrets

Databricks secrets provide a secure way to store credentials and sensitive configuration values, making it easy to protect credentials when running notebooks and jobs. Instead of hardcoding sensitive values, use secret scopes to manage environment variables securely.

### Secret Scopes

A secret scope is a collection of secrets identified by a name. Databricks recommends aligning secret scopes to roles or applications rather than individuals.

📖 **Resource:** [Databricks Secrets management](https://docs.databricks.com/aws/en/security/secrets/)

### Accessing Secrets in Code

Retrieve secret values programmatically using `dbutils`:

<br>

```python
password = dbutils.secrets.get(scope = "<scope-name>", key = "<key-name>")
```

Databricks redacts all secret values read using `dbutils.secrets.get()` - when displayed, secret values are replaced with `[REDACTED]`. 

[**Security Considerations**](https://docs.databricks.com/aws/en/security/secrets/secrets-spark-conf-env-var)

## Monitoring Apps

Apps automatically capture logs written to stdout and stderr, which can be accessed through the Apps UI (via the Logs tab on the app details page) or by appending `/logz` to your app URL.  For example, logs for `https://my-app-123.databricks.com` are available at `https://my-app-123.databricks.com/logz`.

**Important**: Databricks doesn't persist logs when app compute shuts down. For long-term storage, integrate with external logging services or write logs to Unity Catalog volumes or tables. 

### Audit Logging & Security

Databricks captures audit logs for app-related activities in the `system.access.audit` table, allowing you to track user actions, permission changes, app configuration modifications, and security events.  This enables monitoring of login activity, permission changes, and user behavior within apps.

### Cost & Resource Monitoring

Use the `system.billing.usage` table to monitor Databricks Apps costs and resource consumption.  This helps track operational expenses on a daily or monthly basis.



📖 **Resource:** [Monitoring Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/monitor)

## Comprehensive Resource Library

Refer to the [Databricks app-templates](https://github.com/databricks/app-templates/tree/main) repository to see examples of building an app.

### 📚 **Official Documentation**
* [Databricks Apps - Complete Guide](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)
* [App Configuration and Settings](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/configuration)
* [Databricks Apps - Best Practices](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/best-practices#gsc.tab=0)
* [Databricks Secrets Management](https://docs.databricks.com/aws/en/security/secrets/)

### 🛠️ **Framework Documentation**
* [Streamlit Documentation](https://docs.streamlit.io/)
* [Gradio Documentation](https://www.gradio.app/docs/)
* [Dash by Plotly](https://dash.plotly.com/)
* [Flask Documentation](https://flask.palletsprojects.com/)

### 🎓 **Hands-On Tutorials**
* [Build Your First Databricks App](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/get-started)
* [Deploying ML Models as Apps](https://docs.databricks.com/en/machine-learning/model-serving/index.html)

### 🏗️ **Architecture Patterns**
* [Production App Best Practices](https://docs.databricks.com/en/dev-tools/databricks-apps/best-practices.html)
* [Security and Governance for Apps](https://docs.databricks.com/en/security/index.html)